### Example 6.8: 2-Dimensional Newton-Raphson Warmup

As a warmup towards solving Exercise 6.2, implement a 2-Dimensional Newton-Raphson method:

$\vec{f} + \mathbf{J} \Delta \vec{x} = 0$ 

where: 

$\vec{f} = \left(\begin{array}{c} 
f_{1}  \\
f_{2}  \\
\end{array}\right)$, $\Delta \vec{x} = \left(\begin{array}{c} 
\Delta x_{1}  \\
\Delta x_{2}  \\
\end{array}\right)$, $\mathbf{J} = \left(\begin{array}{cccc} 
\frac{ \partial f_1 }{ \partial x_1 } & \frac{ \partial f_1 }{ \partial x_2 }\\
\frac{ \partial f_2 }{ \partial x_1 } & \frac{ \partial f_2 }{ \partial x_2 } 
\end{array}\right)$,

and solve via:  

$\mathbf{J} \Delta \vec{x} = -\vec{f}$.

using the ```linalg``` package.

You will need to: 

- Create a function that calculates the partial derivatives that enter the Jacobian, $\frac{ \partial f_i }{ \partial x_j }$. You may use the central-difference derivative.
- Create a function that calculates the actual Jacobian matrix, calling the aforementioned function.
- Create a function that returns the vector of functions $\vec{f}$.
- Use $\mathbf{J}$ and $-\vec{f}$ to solve the system iteratively for $\Delta \vec{x}$, adding it to $\vec{x}$, until a solution is reached, i.e. $|f_i| \sim 0$ within the required precision. 

(a) Check your Jacobian with the *linear* system:

$2 x + y - 13 = 0$

$x + y - 9 = 0$

(b) Use your code for the 2-D Newton Raphson to solve the linear system above. 

(c) Use your code to solve the nonlinear system:

$x^2 + y - 21 = 0$

$x + y^2 - 29 = 0$

Verify that your solutions correspond to the expected ones ($ x = 4, y = 5$).

(d) Use your code to solve the system:

$xe^y = 1$

$-x^2 + y = 1$

In [4]:
# Example 6.8 in class
import numpy as np

# write the linear function to be used for testing: 
def flinear(x):
    f1 = 2*x[0] + x[1] - 13 
    f2 = x[0] + x[1] - 9
    fvec = [f1, f2] 
    return np.array(fvec)

# STEP 1: partial derivative calculator
def partial_fi_partial_xj(f,x,i, j, h=1E-5):
    """Partial derivative of i-th function with respect to j-th component""" 
    dx = np.zeros(len(x))
    dx[j] += h
    dfidxj = (f(x+dx)[i] - f(x)[i])/h
    return dfidxj 

# STEP 2: Use the partial derivative function to alculate the Jacobian
def jacobian(f,x,h=1E-5):
    """This function assembles the Jacobian matrix"""
    # calculate the number of unknowns:
    N = len(x)
    jac = np.zeros((N,N))
    for i in range(N):
        for j in range(N):
            jac[i][j] = partial_fi_partial_xj(f, x, i, j)
    return jac

# checks: 
# derivative df0/dx0 (=2)
xtest = [1,1]
df1dx1_linear_test = partial_fi_partial_xj(flinear, xtest, 0, 0)
print("Consistency test 1: df0/dx0 =", df1dx1_linear_test)
# test Jacobian for linear function:
jac_linear_test = jacobian(flinear, xtest)
print('Jacobian test for the linear function=', jac_linear_test)

# now solve using linalg's solve:
f_at_xtest = flinear(xtest)
Dxsol_test = np.linalg.solve(jac_linear_test, -f_at_xtest)
print('Solutions to linear problem=', Dxsol_test+xtest)

Consistency test 1: df0/dx0 = 1.9999999999242843
Jacobian test for the linear function= [[2. 1.]
 [1. 1.]]
Solutions to linear problem= [4. 5.]


In [7]:
def NewtonRaphsonND(f, x0,  Nmax, prec, h=1E-5):
    """This function implements the Newton-Rapshon method in N dimensions"""
    N = len(f(x0))
    for nstep in range(Nmax):
        J = jacobian(f,x0,h=h)
        minus_f = - f(x0)
        Dx = np.linalg.solve(J,minus_f)
        x0 = x0 + Dx
        # loop: check that all fi's are below prec (otherwise continue the loop)
        #reached_precision = True
        #for fi in f(x0):
        #    if abs(fi) > prec:
        #        reached_precision = False
        #if reached_precision is True:
        #    break
        if np.all(np.abs(f(x0)) < prec):
            print('Newton-Raphson desired precision reached')
            break
    return x0, nstep
            
Nmax=1000
x0 = np.array([1,1])
prec = 1E-10
xsol, nsteps = NewtonRaphsonND(flinear, x0,  Nmax, prec, h=1E-5)
print('Solution to linear system=', xsol, 'after', nsteps, 'steps')

Newton-Raphson desired precision reached
Solution to linear system= [4. 5.] after 1 steps


In [10]:
# part (c): "easy" nonlinear problem: 
def fnonlinear1(x):
    f1 = x[0]**2 + x[1] - 21 
    f2 = x[0] + x[1]**2 - 29
    fvec = [f1, f2] 
    return np.array(fvec)

Nmax=1000
x0 = np.array([0,0])
prec = 1E-10
xsol, nsteps = NewtonRaphsonND(fnonlinear1, x0,  Nmax, prec, h=1E-5)
print('Solution to nonlinear system (part c)=', xsol, 'after', nsteps, 'steps')


Newton-Raphson desired precision reached
Solution to nonlinear system (part c)= [4. 5.] after 7 steps


In [13]:
# part (d): "challenging" nonlinear problem: 
def fnonlinear2(x):
    f1 = x[0] * np.exp(x[1]) - 1
    f2 = - x[0]**2 + x[1] - 1
    fvec = [f1, f2] 
    return np.array(fvec)

Nmax=1000
x0 = np.array([0,0])
prec = 1E-10
xsol, nsteps = NewtonRaphsonND(fnonlinear2, x0,  Nmax, prec, h=1E-5)
print('Solution to nonlinear system (part c)=', xsol, 'after', nsteps, 'steps')

# check if fis are zero for this solution

print('Function for part d evaluated at xsol=', fnonlinear2(xsol))

Newton-Raphson desired precision reached
Solution to nonlinear system (part c)= [0.32993568 1.10885755] after 5 steps
Function for part d evaluated at xsol= [-1.6875390e-13 -3.3462122e-13]
